<a href="https://colab.research.google.com/github/benasphy/DecisionTree/blob/main/Decision_Tree_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        """Build the decision tree."""
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        num_labels = len(np.unique(y))

        # stopping conditions
        if (depth >= self.max_depth or num_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return {"leaf": True, "value": leaf_value}

        # find the best split
        best_feature, best_thresh = self._best_split(X, y)
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return {"leaf": True, "value": leaf_value}

        # split the dataset
        left_idx = X[:, best_feature] <= best_thresh
        right_idx = X[:, best_feature] > best_thresh

        # recursive building
        left_subtree = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right_subtree = self._build_tree(X[right_idx], y[right_idx], depth + 1)

        return {
            "leaf": False,
            "feature": best_feature,
            "threshold": best_thresh,
            "left": left_subtree,
            "right": right_subtree
        }

    def _best_split(self, X, y):
        """Find the best split (feature, threshold) that minimizes Gini impurity."""
        n_samples, n_features = X.shape
        best_gini = float("inf")
        best_feature, best_thresh = None, None

        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_idx = X[:, feature_idx] <= threshold
                right_idx = X[:, feature_idx] > threshold
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue
                gini = self._gini_split(y[left_idx], y[right_idx])
                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature_idx
                    best_thresh = threshold

        return best_feature, best_thresh

    def _gini(self, y):
        """Compute Gini impurity."""
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)

    def _gini_split(self, left_y, right_y):
        """Weighted Gini impurity after split."""
        n = len(left_y) + len(right_y)
        gini_left = self._gini(left_y)
        gini_right = self._gini(right_y)
        weighted = (len(left_y) / n) * gini_left + (len(right_y) / n) * gini_right
        return weighted

    def _most_common_label(self, y):
        """Return the most frequent label."""
        return np.bincount(y).argmax()

    def predict(self, X):
        """Predict class labels for samples."""
        return np.array([self._predict_single(x, self.tree) for x in X])

    def _predict_single(self, x, tree):
        if tree["leaf"]:
            return tree["value"]
        feature_val = x[tree["feature"]]
        if feature_val <= tree["threshold"]:
            return self._predict_single(x, tree["left"])
        else:
            return self._predict_single(x, tree["right"])


In [2]:
# Sample dataset
X = np.array([[2, 3],
              [1, 1],
              [3, 2],
              [6, 5],
              [7, 8],
              [8, 6]])
y = np.array([0, 0, 0, 1, 1, 1])

# Train the decision tree
tree = DecisionTree(max_depth=3)
tree.fit(X, y)

# Predict
preds = tree.predict(np.array([[2, 2], [7, 7]]))
print("Predictions:", preds)


Predictions: [0 1]
